<a href="https://colab.research.google.com/github/abhiramanil369/VTP-Deep-Learning-for-Vehicle-Trajectory-Prediction/blob/main/VTP_Deep_Learning_for_Vehicle_Trajectory_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/leilin-research/VTP.git
%cd VTP

Cloning into 'VTP'...
remote: Enumerating objects: 920, done.
remote: Counting objects: 100% (215/215), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 920 (delta 48), reused 189 (delta 28), pack-reused 705 (from 1)
Receiving objects: 100% (920/920), 18.86 MiB | 14.73 MiB/s, done.
Resolving deltas: 100% (398/398), done.
/content/VTP


In [2]:
!ls

Attention-LSTM	README.md  requirements.txt  STA-LSTM


In [3]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [4]:
!pip install torch torchvision numpy pandas matplotlib scipy tqdm

In [5]:
import torch
torch.cuda.is_available()

True

In [6]:
import kagglehub

path = kagglehub.dataset_download("nigelwilliams/ngsim-vehicle-trajectory-data-us-101")

print("Path to dataset files:", path)

100%|██████████| 38.5M/38.5M [00:00<00:00, 65.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nigelwilliams/ngsim-vehicle-trajectory-data-us-101/versions/2


In [7]:
import shutil
import os

destination = "/content/VTP/data/ngsim"

os.makedirs(destination, exist_ok=True)

shutil.copytree(path, destination, dirs_exist_ok=True)

'/content/VTP/data/ngsim'

In [8]:
!ls data/ngsim

data-analysis-report-0750-0805.pdf  trajectory-data-dictionary.htm
trajectories-0750am-0805am.txt


In [9]:
import pandas as pd

file_path = "/content/VTP/data/ngsim/trajectories-0750am-0805am.txt"

df = pd.read_csv(file_path, sep=r"\s+", engine="python")
df.head()

,2,13,437,1118846980200,16.467,35.381,6451137.641,1873344.962,14.5,4.9,2.1,40.00,0.00,2.2,0,0.1,0.00.1,0.00.2
0,2,14,437,1118846980300,16.447,39.381,6451140.329,1873342.000,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
1,2,15,437,1118846980400,16.426,43.381,6451143.018,1873339.038,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
2,2,16,437,1118846980500,16.405,47.380,6451145.706,1873336.077,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
3,2,17,437,1118846980600,16.385,51.381,6451148.395,1873333.115,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
4,2,18,437,1118846980700,16.364,55.381,6451151.084,1873330.153,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0


In [10]:
df.columns

Index(['2', '13', '437', '1118846980200', '16.467', '35.381', '6451137.641',
       '1873344.962', '14.5', '4.9', '2.1', '40.00', '0.00', '2.2', '0', '0.1',
       '0.00.1', '0.00.2'],
      dtype='object')

In [11]:
import pandas as pd

file_path = "/content/VTP/data/ngsim/trajectories-0750am-0805am.txt"

df = pd.read_csv(
    file_path,
    sep=r"\s+",
    header=None,
    engine="python"
)

df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,2,13,437,1118846980200,16.467,35.381,6451137.641,1873344.962,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
1,2,14,437,1118846980300,16.447,39.381,6451140.329,1873342.000,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
2,2,15,437,1118846980400,16.426,43.381,6451143.018,1873339.038,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
3,2,16,437,1118846980500,16.405,47.380,6451145.706,1873336.077,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
4,2,17,437,1118846980600,16.385,51.381,6451148.395,1873333.115,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0


In [12]:
df.columns

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17], dtype='int64')

In [13]:
df.columns = [
    "Vehicle_ID",        # 0
    "Frame_ID",          # 1
    "Total_Frames",      # 2
    "Global_Time",       # 3
    "Local_X",           # 4
    "Local_Y",           # 5
    "Global_X",          # 6
    "Global_Y",          # 7
    "Vehicle_Length",    # 8
    "Vehicle_Width",     # 9
    "Vehicle_Class",     # 10
    "Vehicle_Velocity",  # 11
    "Vehicle_Acceleration", # 12
    "Lane_ID",           # 13
    "Preceding",         # 14
    "Following",         # 15
    "Space_Headway",     # 16
    "Time_Headway"       # 17
]

In [14]:
df.head()

,Vehicle_ID,Frame_ID,Total_Frames,Global_Time,Local_X,Local_Y,Global_X,Global_Y,Vehicle_Length,Vehicle_Width,Vehicle_Class,Vehicle_Velocity,Vehicle_Acceleration,Lane_ID,Preceding,Following,Space_Headway,Time_Headway
0,2,13,437,1118846980200,16.467,35.381,6451137.641,1873344.962,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
1,2,14,437,1118846980300,16.447,39.381,6451140.329,1873342.000,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
2,2,15,437,1118846980400,16.426,43.381,6451143.018,1873339.038,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
3,2,16,437,1118846980500,16.405,47.380,6451145.706,1873336.077,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0
4,2,17,437,1118846980600,16.385,51.381,6451148.395,1873333.115,14.5,4.9,2,40.0,0.0,2,0,0,0.0,0.0


In [15]:
df["Local_X"] *= 0.3048
df["Local_Y"] *= 0.3048

In [16]:
df = df.sort_values(by=["Vehicle_ID", "Frame_ID"])

In [17]:
df.to_csv(
    "/content/VTP/data/ngsim/processed_ngsim.txt",
    sep=" ",
    header=False,
    index=False
)

In [18]:
df.isnull().sum()

,0
Vehicle_ID,0
Frame_ID,0
Total_Frames,0
Global_Time,0
Local_X,0
Local_Y,0
Global_X,0
Global_Y,0
Vehicle_Length,0
Vehicle_Width,0


In [19]:
df = df.dropna()

In [20]:
# Take first 300 vehicles only
vehicle_ids = df["Vehicle_ID"].unique()[:300]
df_small = df[df["Vehicle_ID"].isin(vehicle_ids)]

In [21]:
df_small = df_small[
    ["Vehicle_ID", "Frame_ID", "Local_X", "Local_Y",
     "Vehicle_Velocity", "Lane_ID"]
]

In [22]:
df_small = df_small.sort_values(by=["Vehicle_ID", "Frame_ID"])

In [23]:
df_small.to_csv(
    "/content/VTP/data/ngsim/baseline_data.txt",
    sep=" ",
    header=False,
    index=False
)

In [24]:
import numpy as np

HISTORY = 30
FUTURE = 50

sequences = []

for vid in df_small["Vehicle_ID"].unique():
    vehicle_data = df_small[df_small["Vehicle_ID"] == vid]
    vehicle_data = vehicle_data.sort_values("Frame_ID")

    coords = vehicle_data[["Local_X", "Local_Y"]].values

    for i in range(len(coords) - HISTORY - FUTURE):
        hist = coords[i:i+HISTORY]
        fut = coords[i+HISTORY:i+HISTORY+FUTURE]

        sequences.append((hist, fut))

print("Total sequences:", len(sequences))

Total sequences: 112530


In [25]:
!ls

Attention-LSTM	data  README.md  requirements.txt  STA-LSTM


In [26]:
%cd Attention-LSTM
!ls

/content/VTP/Attention-LSTM
evaluate.py  images  model.py  README.md  train.py  utils.py


In [27]:
!sed -n '1,200p' train.py

from __future__ import print_function
import torch
from model import highwayNet
from utils import ngsimDataset,maskedMSE
from torch.utils.data import DataLoader
import time
import math
import datetime
import numpy as np
import random
import os


## Network Arguments
args = {}
args['use_cuda'] = True
args['encoder_size'] = 64 # lstm encoder hidden state size, adjustable
args['decoder_size'] = 128 # lstm decoder  hidden state size, adjustable
args['in_length'] = 16
args['out_length'] = 20
args['grid_size'] = (265,3) # (660/5 *2 +1 )*3
args['input_embedding_size'] = 32 # input dimension for lstm encoder, adjustable
args['train_flag'] = True




def seed_torch(seed=1029):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you are using multi-GPU.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_torch()


In [28]:
!ls ../../data/trajectory

ls: cannot access '../../data/trajectory': No such file or directory


In [29]:
import pandas as pd

file_path = "/content/trajectories-0750am-0805am.txt"

cols = [
    "Vehicle_ID", "Frame_ID", "Total_Frames", "Global_Time",
    "Local_X", "Local_Y", "Global_X", "Global_Y",
    "v_Length", "v_Width", "v_Class",
    "v_Vel", "v_Acc", "Lane_ID",
    "Preceding", "Following",
    "Space_Headway", "Time_Headway"
]

df = pd.read_csv(file_path, sep=r"\s+", header=None)
df.columns = cols

print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/trajectories-0750am-0805am.txt'

In [30]:
!find /content -name "trajectories-0750am-0805am.txt"

/content/VTP/data/ngsim/trajectories-0750am-0805am.txt


In [31]:
path = kagglehub.dataset_download("nigelwilliams/ngsim-vehicle-trajectory-data-us-101")
print(path)

Using Colab cache for faster access to the 'ngsim-vehicle-trajectory-data-us-101' dataset.
/kaggle/input/ngsim-vehicle-trajectory-data-us-101


In [32]:
import os
print(os.listdir(path))

['data-analysis-report-0750-0805.pdf', 'trajectories-0750am-0805am.txt', 'trajectory-data-dictionary.htm']


In [33]:
file_path = os.path.join(path, "trajectories-0750am-0805am.txt")
print(file_path)

/kaggle/input/ngsim-vehicle-trajectory-data-us-101/trajectories-0750am-0805am.txt


In [34]:
df = pd.read_csv(file_path, sep=r"\s+", header=None)

In [35]:
import kagglehub
import os
import pandas as pd

# Download dataset
path = kagglehub.dataset_download("nigelwilliams/ngsim-vehicle-trajectory-data-us-101")

# Check files
print("Dataset folder:", path)
print("Files:", os.listdir(path))

# Build correct file path
file_path = os.path.join(path, "trajectories-0750am-0805am.txt")

# Load
df = pd.read_csv(file_path, sep=r"\s+", header=None)

print(df.head())

Using Colab cache for faster access to the 'ngsim-vehicle-trajectory-data-us-101' dataset.
Dataset folder: /kaggle/input/ngsim-vehicle-trajectory-data-us-101
Files: ['data-analysis-report-0750-0805.pdf', 'trajectories-0750am-0805am.txt', 'trajectory-data-dictionary.htm']
   0   1    2              3       4       5            6            7     8   \
0   2  13  437  1118846980200  16.467  35.381  6451137.641  1873344.962  14.5   
1   2  14  437  1118846980300  16.447  39.381  6451140.329  1873342.000  14.5   
2   2  15  437  1118846980400  16.426  43.381  6451143.018  1873339.038  14.5   
3   2  16  437  1118846980500  16.405  47.380  6451145.706  1873336.077  14.5   
4   2  17  437  1118846980600  16.385  51.381  6451148.395  1873333.115  14.5   

    9   10    11   12  13  14  15   16   17  
0  4.9   2  40.0  0.0   2   0   0  0.0  0.0  
1  4.9   2  40.0  0.0   2   0   0  0.0  0.0  
2  4.9   2  40.0  0.0   2   0   0  0.0  0.0  
3  4.9   2  40.0  0.0   2   0   0  0.0  0.0  
4  4.9   2 

In [36]:
df.shape

(1180598, 18)

In [37]:
cols = [
    "Vehicle_ID", "Frame_ID", "Total_Frames", "Global_Time",
    "Local_X", "Local_Y", "Global_X", "Global_Y",
    "v_Length", "v_Width", "v_Class",
    "v_Vel", "v_Acc", "Lane_ID",
    "Preceding", "Following",
    "Space_Headway", "Time_Headway"
]

df.columns = cols
print(df.head())

   Vehicle_ID  Frame_ID  Total_Frames    Global_Time  Local_X  Local_Y  \
0           2        13           437  1118846980200   16.467   35.381   
1           2        14           437  1118846980300   16.447   39.381   
2           2        15           437  1118846980400   16.426   43.381   
3           2        16           437  1118846980500   16.405   47.380   
4           2        17           437  1118846980600   16.385   51.381   

      Global_X     Global_Y  v_Length  v_Width  v_Class  v_Vel  v_Acc  \
0  6451137.641  1873344.962      14.5      4.9        2   40.0    0.0   
1  6451140.329  1873342.000      14.5      4.9        2   40.0    0.0   
2  6451143.018  1873339.038      14.5      4.9        2   40.0    0.0   
3  6451145.706  1873336.077      14.5      4.9        2   40.0    0.0   
4  6451148.395  1873333.115      14.5      4.9        2   40.0    0.0   

   Lane_ID  Preceding  Following  Space_Headway  Time_Headway  
0        2          0          0            0.0     

In [38]:
df = df[["Vehicle_ID", "Frame_ID", "Local_X", "Local_Y", "Lane_ID"]]

In [39]:
df = df.sort_values(by=["Vehicle_ID", "Frame_ID"])

In [40]:
print("Unique vehicles:", df["Vehicle_ID"].nunique())
print("Min frames per vehicle:", df.groupby("Vehicle_ID").size().min())
print("Max frames per vehicle:", df.groupby("Vehicle_ID").size().max())

Unique vehicles: 2169
Min frames per vehicle: 177
Max frames per vehicle: 1010


In [41]:
import numpy as np

t_h = 30
t_f = 50

samples = []

for veh_id in df["Vehicle_ID"].unique():
    veh_data = df[df["Vehicle_ID"] == veh_id]

    if len(veh_data) < t_h + t_f:
        continue

    veh_data = veh_data.sort_values("Frame_ID")

    coords = veh_data[["Local_X","Local_Y"]].values

    for i in range(len(coords) - t_h - t_f):
        hist = coords[i:i+t_h]
        fut = coords[i+t_h:i+t_h+t_f]
        samples.append((hist, fut))

print("Total samples created:", len(samples))

Total samples created: 1007078


In [42]:
histories = np.array([s[0] for s in samples])
futures = np.array([s[1] for s in samples])

print("Hist shape:", histories.shape)
print("Future shape:", futures.shape)

Hist shape: (1007078, 30, 2)
Future shape: (1007078, 50, 2)


In [43]:
# OPTIONAL: reduce dataset size for baseline experiment
max_samples = 200000

histories = histories[:max_samples]
futures = futures[:max_samples]

print(histories.shape)

(200000, 30, 2)


In [44]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    histories,
    futures,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (160000, 30, 2)
Val: (40000, 30, 2)


In [45]:
!pip install scipy

In [46]:
from scipy.io import savemat
import os

os.makedirs("/content/VTP/data/trajectory", exist_ok=True)

savemat("/content/VTP/data/trajectory/TrainSet_us101.mat", {
    "hist": X_train,
    "fut": y_train
})

savemat("/content/VTP/data/trajectory/ValSet_us101.mat", {
    "hist": X_val,
    "fut": y_val
})

print("MAT files saved successfully.")

MAT files saved successfully.


In [47]:
!grep -n "class ngsimDataset" utils.py

14:class ngsimDataset(Dataset):


In [48]:
!sed -n '1,200p' utils.py

from __future__ import print_function, division
from torch.utils.data import Dataset, DataLoader
import scipy.io as scp
import numpy as np
import torch
import h5py
import hdf5storage
import random
import os

#___________________________________________________________________________________________________________________________

### Dataset class for the NGSIM dataset
class ngsimDataset(Dataset):

    def __seed_torch(seed=1029):
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # if you are using multi-GPU.
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


    def __init__(self, mat_file, t_h=30, t_f=40, d_s=2, enc_size = 64, grid_size = (265,3), CAV_ratio = 0.7):
        i = 0
        print (mat_file)
        self.D = hdf5storage.loadmat(mat_file)['res_traj'] 
        pri

In [49]:
from torch.utils.data import Dataset
import torch
import scipy.io as scp

class SimpleDataset(Dataset):
    def __init__(self, mat_file):
        data = scp.loadmat(mat_file)
        self.hist = data['hist']
        self.fut = data['fut']

    def __len__(self):
        return len(self.hist)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.hist[idx]), torch.FloatTensor(self.fut[idx])

In [50]:
import torch
from torch.utils.data import Dataset, DataLoader
import scipy.io as scp

In [51]:
class SimpleDataset(Dataset):
    def __init__(self, mat_file):
        data = scp.loadmat(mat_file)
        self.hist = data['hist']
        self.fut = data['fut']

    def __len__(self):
        return len(self.hist)

    def __getitem__(self, idx):
      hist = torch.FloatTensor(self.hist[idx])
      fut = torch.FloatTensor(self.fut[idx])

      ref = hist[-1].clone()
      hist = hist - ref
      fut = fut - ref

      return hist, fut

In [52]:
train_dataset = SimpleDataset("/content/VTP/data/trajectory/TrainSet_us101.mat")

print("Dataset size:", len(train_dataset))

sample_hist, sample_fut = train_dataset[0]

print("Hist shape:", sample_hist.shape)
print("Future shape:", sample_fut.shape)

Dataset size: 160000
Hist shape: torch.Size([30, 2])
Future shape: torch.Size([50, 2])


In [65]:
import torch
from torch.utils.data import Dataset, DataLoader
import scipy.io as scp

class SimpleDataset(Dataset):
    def __init__(self, mat_file):
        data = scp.loadmat(mat_file)
        self.hist = data['hist']
        self.fut = data['fut']

    def __len__(self):
        return len(self.hist)

    def __getitem__(self, idx):
        hist = torch.FloatTensor(self.hist[idx])
        fut = torch.FloatTensor(self.fut[idx])

        hist = hist * 0.3048
        fut = fut * 0.3048

        ref = hist[-1].clone()
        hist = hist - ref
        fut = fut - ref

        return hist, fut

In [66]:
train_dataset = SimpleDataset("/content/VTP/data/trajectory/TrainSet_us101.mat")
val_dataset   = SimpleDataset("/content/VTP/data/trajectory/ValSet_us101.mat")

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False)

print("Train batches:", len(train_loader))

Train batches: 1250


In [61]:
import torch.nn as nn

class TrajectoryLSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, output_size=2, pred_len=50):
        super(TrajectoryLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.pred_len = pred_len

        self.encoder = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(output_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, hist):

        batch_size = hist.size(0)

        # Encode history
        _, (hidden, cell) = self.encoder(hist)

        # Start decoder with last position
        decoder_input = hist[:, -1:, :]  # last observed position

        outputs = []

        for _ in range(self.pred_len):
            out, (hidden, cell) = self.decoder(decoder_input, (hidden, cell))
            pred = self.fc(out)
            outputs.append(pred)
            decoder_input = pred

        outputs = torch.cat(outputs, dim=1)
        return outputs

In [67]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TrajectoryLSTM().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Using device:", device)

Using device: cuda


In [68]:
def calculate_metrics(pred, gt):
    # pred, gt: (batch, 50, 2)

    ade = torch.mean(torch.norm(pred - gt, dim=2))
    fde = torch.mean(torch.norm(pred[:, -1, :] - gt[:, -1, :], dim=1))

    return ade.item(), fde.item()

In [70]:
hist, fut = train_dataset[0]
print("Max value hist:", hist.abs().max())
print("Max value fut:", fut.abs().max())

Max value hist: tensor(36.9259)
Max value fut: tensor(67.2185)


In [71]:
epochs = 10

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for hist, fut in train_loader:
        hist = hist.to(device)
        fut = fut.to(device)

        optimizer.zero_grad()
        output = model(hist)

        loss = criterion(output, fut)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    total_ade = 0
    total_fde = 0

    with torch.no_grad():
        for hist, fut in val_loader:
            hist = hist.to(device)
            fut = fut.to(device)

            output = model(hist)
            ade, fde = calculate_metrics(output, fut)

            total_ade += ade
            total_fde += fde

    avg_ade = total_ade / len(val_loader)
    avg_fde = total_fde / len(val_loader)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {avg_loss:.4f}")
    print(f"Val ADE: {avg_ade:.4f}")
    print(f"Val FDE: {avg_fde:.4f}")
    print("-"*40)

Epoch 1/10
Train Loss: 68.1609
Val ADE: 3.2561
Val FDE: 14.2260
----------------------------------------
Epoch 2/10
Train Loss: 12.0178
Val ADE: 2.3228
Val FDE: 7.9025
----------------------------------------
Epoch 3/10
Train Loss: 6.3844
Val ADE: 2.1150
Val FDE: 6.0917
----------------------------------------
Epoch 4/10
Train Loss: 5.2095
Val ADE: 1.9377
Val FDE: 4.9991
----------------------------------------
Epoch 5/10
Train Loss: 4.8673
Val ADE: 1.9053
Val FDE: 4.6648
----------------------------------------
Epoch 6/10
Train Loss: 4.7135
Val ADE: 1.9328
Val FDE: 4.5747
----------------------------------------
Epoch 7/10
Train Loss: 4.6929
Val ADE: 1.9015
Val FDE: 4.4521
----------------------------------------
Epoch 8/10
Train Loss: 4.5913
Val ADE: 1.8691
Val FDE: 4.3895
----------------------------------------
Epoch 9/10
Train Loss: 4.5838
Val ADE: 1.8826
Val FDE: 4.3900
----------------------------------------
Epoch 10/10
Train Loss: 4.5730
Val ADE: 1.9345
Val FDE: 4.5380
-------